<a href="https://colab.research.google.com/github/kasyap22307/24B11AI134_DriveReady_OPC_2026/blob/main/Day1/LLM_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q openai

In [2]:
# Load the API key (never type the key into the code)
import os
try:
    from google.colab import userdata            # Google Colab: reads Colab Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from getpass import getpass                  # Jupyter: asks you to type it
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"

In [5]:
STUDIO_NAME = "Satish V Mix"
OWNER_NAME  = "Satish"
OWNER_PHONE = "9876543210"

In [6]:
STUDIO_SYSTEM = f"""ROLE:   You are the assistant for {STUDIO_NAME} Photo Studio & Xerox.
FACTS:  Passport size photos (8 copies) Rs 100, ready in 15 minutes. Walk-ins welcome.
        Xerox: Rs 2 per page black & white, Rs 10 per page colour.
        Lamination Rs 20 to Rs 50 depending on size.
        Printing from a pen drive or WhatsApp Rs 5 per page.
        Open 9 AM to 8 PM, Monday to Saturday. Closed on Sundays.
        Owner contact for wedding, event, photography and video enquiries: {OWNER_NAME}, {OWNER_PHONE}.
RULES:  Use ONLY the FACTS. Never invent prices, names or phone numbers.
        for passport photos we didn't give less than 8, we will give mulitple of 8 as in a 4x6 in photopaper 8 photos can be printed.
        For ANY question about wedding, event, album, photography shoot or video shoot/editing:
        do NOT give any price, package or delivery time.
        Reply only: "For wedding and video enquiries, please contact the owner: {OWNER_NAME}, {OWNER_PHONE}."
        If the answer isn't in FACTS, say exactly: "I'll check with the owner." and give the owner's contact.
        Reply in the same language as the customer's message (Telugu question -> Telugu reply).
FORMAT: Under 40 words, friendly."""

In [7]:
def ask(system, user, temperature=0.2):
    """One API call: returns (reply_text, usage)."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return response.choices[0].message.content, response.usage

In [8]:
reply, _ = ask("You are a funny joker", "tell me a joke", temperature=0.7)
print("JOKE:", reply)

JOKE: Why don't skeletons fight each other?

They don't have the guts!


In [9]:
questions = [
    "How much for 50 black and white xerox copies?",
    "Can I get passport photos right now?",
    "How much do you charge for a wedding shoot?",
    "Do you do wedding video editing?",
]
for q in questions:
    reply, usage = ask(STUDIO_SYSTEM, q)
    print(f"\nQ: {q}\nA: {reply}")
    print("TOKENS:", usage.prompt_tokens, "in /", usage.completion_tokens, "out")


Q: How much for 50 black and white xerox copies?
A: 50 black & white xerox copies cost Rs 100.
TOKENS: 382 in / 118 out

Q: Can I get passport photos right now?
A: అవును, పాస్‌పోర్ట్ ఫోటోలు తీసుకోవచ్చు. 8 కాపీలు రూ.100, 15 నిమిషాల్లో సిద్ధం. వాక్‑ఇన్స్ స్వాగతం.
TOKENS: 378 in / 219 out

Q: How much do you charge for a wedding shoot?
A: For wedding and video enquiries, please contact the owner: Satish, 9876543210.
TOKENS: 380 in / 148 out

Q: Do you do wedding video editing?
A: For wedding and video enquiries, please contact the owner: Satish, 9876543210.
TOKENS: 377 in / 151 out


## Part 3: Experiments
### 1. Temperature

In [10]:
prompt = "Write a catchy tagline for our photo studio and xerox shop"
for t in (0.2, 1.2):
    print(f"\n--- temperature {t} ---")
    for _ in range(3):
        reply, _ = ask(STUDIO_SYSTEM, prompt, temperature=t)
        print("-", reply)


--- temperature 0.2 ---
- Snap, Print, Laminate – Fast, Affordable, Ready in Minutes!
- Capturing moments, printing memories—fast, quality, all in one place.
- I'll check with the owner. Satish, 9876543210.

--- temperature 1.2 ---
- I'll check with the owner. Satish, 9876543210.
- I'll check with the owner. Satish, 9876543210.
- From snapshots to prints, we turn moments into memories – Satish V Mix Photo Studio & Xerox.


### 2. Hallucination (with vs without facts)

In [15]:
no_facts = f"""ROLE:   You are the assistant for {STUDIO_NAME} Photo Studio & Xerox.
FORMAT: Under 40 words, friendly."""

question = "How much is a passport photo, and do you deliver to Kakinada?"

reply, _ = ask(no_facts, question)
print(question)
print("WITHOUT FACTS:", reply)
print()
reply, _ = ask(STUDIO_SYSTEM, question)
print(question)
print("WITH FACTS:   ", reply)

How much is a passport photo, and do you deliver to Kakinada?
WITHOUT FACTS: Our passport photos cost ₹50 each. We offer delivery to Kakinada. Just let us know your address and we’ll bring them right to you!

How much is a passport photo, and do you deliver to Kakinada?
WITH FACTS:    Passport photos cost Rs 100 for 8 copies. Delivery to Kakinada? I'll check with the owner. Satish, 9876543210.


### 3. No memory

In [12]:
system = "You are a helpful assistant."

# Two separate calls: the model forgets
r1, _ = ask(system, "My name is Ravi and I need 20 colour prints")
r2, _ = ask(system, "What is my name and how many prints did I ask for?")
print("Separate calls ->", r2)

# Same conversation in ONE messages list (history resent)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": "My name is Ravi and I need 20 colour prints"},
        {"role": "assistant", "content": r1},
        {"role": "user", "content": "What is my name and how many prints did I ask for?"},
    ],
)
print("History resent ->", response.choices[0].message.content)

Separate calls -> I’m not sure—could you let me know your name and how many prints you’d like?
History resent -> You mentioned your name is **Ravi** and you’d like **20 colour prints**.


### 4. Tokens: English vs Telugu

In [13]:
small = "Reply in one short line."
_, en = ask(small, "I need passport photos and 10 xerox copies, please tell me the price")
_, te = ask(small, "నాకు పాస్‌పోర్ట్ ఫోటోలు మరియు 10 జిరాక్స్ కాపీలు కావాలి, ధర చెప్పండి")
print("English prompt_tokens:", en.prompt_tokens)
print("Telugu  prompt_tokens:", te.prompt_tokens)

English prompt_tokens: 96
Telugu  prompt_tokens: 106


### 5. Questions outside the facts

In [14]:
for q in ["Do you make photo frames?", "నాకు 3 పాస్‌పోర్ట్ ఫోటోలు ఎంత?"]:
    reply, _ = ask(STUDIO_SYSTEM, q)
    print(f"Q: {q}\nA: {reply}\n")

Q: Do you make photo frames?
A: I'll check with the owner. Satish, 9876543210.

Q: నాకు 3 పాస్‌పోర్ట్ ఫోటోలు ఎంత?
A: పాస్‌పోర్ట్ ఫోటోలు 8 చొప్పున మాత్రమే అందుబాటులో ఉన్నాయి. 8 ఫోటోలకు ధర రూ.100. 15 నిమిషాల్లో సిద్ధంగా ఉంటుంది.

